In [ ]:
from google.colab import files
files.upload()

In [ ]:
import shutil, os
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json','/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
#ROBOFLOW_URL='https://app.roboflow.com/dinakar-project/e-waste-dataset-r0ojc-cmavz/1/download/coco?api_key=rf_aXxTDG9jO7XmE6JjZlzmCsKhdfq2'

In [ ]:
ROBOFLOW_URL = 'https://app.roboflow.com/ds/aQTNWqXzY7?key=su7CFZpOed'

In [ ]:
KAGGLE_SLUG="akshat103/e-waste-image-dataset"
!kaggle datasets download -d {KAGGLE_SLUG} -p /content/data/kaggle --unzip

In [ ]:
import os
print(os.listdir('/content/data/kaggle/modified-dataset')[:50])

In [ ]:
import requests, zipfile, io, os
r = requests.get(ROBOFLOW_URL, stream=True)
print("status", r.status_code, "content-type", r.headers.get('content-type'))
if r.status_code != 200:
    print("Download failed; server responded:", r.status_code)
    print(r.text[:800])
else:
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("/content/data/roboflow")
    print("Roboflow extracted to /content/data/roboflow")
    print(os.listdir('/content/data/roboflow')[:50])

In [ ]:
def ls_tree(path, max_levels=2, max_items=20):
    import os
    print(path)
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > max_levels:
            continue
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/ -> {len(files)} files")
        if len(files) > 0:
            for f in files[:min(len(files), max_items)]:
                print(f"{indent}  - {f}")
    print()

ls_tree('/content/data/kaggle')
ls_tree('/content/data/roboflow')

In [ ]:
import os, shutil, hashlib, json
from PIL import Image
from tqdm import tqdm

KAGGLE_ROOT = '/content/data/kaggle'
ROBOFLOW_ROOT = '/content/data/roboflow'
COMBINED_ROOT = '/content/data/combined'
os.makedirs(COMBINED_ROOT, exist_ok=True)

In [ ]:
seen_hashes = set()
def sha256(fpath):
    h = hashlib.sha256()
    with open(fpath,'rb') as f:
        while True:
            ch = f.read(8192)
            if not ch:
                break
            h.update(ch)
    return h.hexdigest()

In [ ]:
def copy_unique(src, dest_dir):
    try:
        h = sha256(src)
    except Exception as e:
        print("hash error", src, e)
        return False
    if h in seen_hashes:
        return False
    seen_hashes.add(h)
    os.makedirs(dest_dir, exist_ok=True)
    base = os.path.basename(src)
    dest = os.path.join(dest_dir, base)
    i = 1
    while os.path.exists(dest):
        name, ext = os.path.splitext(base)
        dest = os.path.join(dest_dir, f"{name}_{i}{ext}")
        i += 1
    shutil.copy2(src, dest)
    return True

In [ ]:
for root, dirs, files in os.walk(KAGGLE_ROOT):
    image_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(image_files) == 0:
        continue
    folder_name = os.path.basename(root)
    if folder_name.lower() in ('annotations','__macosx'):
        continue
    dest_dir = os.path.join(COMBINED_ROOT, folder_name)
    for f in tqdm(image_files, desc=f"Ingest Kaggle {folder_name}"):
        copy_unique(os.path.join(root,f), dest_dir)

In [ ]:
coco_json = os.path.join(ROBOFLOW_ROOT, 'annotations', 'instances_default.json')
if os.path.exists(coco_json):
    with open(coco_json,'r') as f:
        coco = json.load(f)
    id2cat = {c['id']: c['name'] for c in coco.get('categories', [])}
    images_info = {img['id']: img for img in coco.get('images', [])}
    for ann in tqdm(coco.get('annotations', []), desc='Ingest Roboflow COCO'):
        imginfo = images_info.get(ann['image_id'])
        if imginfo is None:
            continue
        fname = imginfo['file_name']
        # try standard locations
        candidate = None
        for sub in ('train','valid','test','images'):
            p = os.path.join(ROBOFLOW_ROOT, sub, fname)
            if os.path.exists(p):
                candidate = p
                break
        if candidate is None:
            # try top-level
            p = os.path.join(ROBOFLOW_ROOT, fname)
            if os.path.exists(p):
                candidate = p
        if candidate is None:
            # skip if image not found
            continue
        cat = id2cat.get(ann['category_id'],'unknown')
        copy_unique(candidate, os.path.join(COMBINED_ROOT, cat))

In [ ]:
data_yaml = os.path.join(ROBOFLOW_ROOT, 'data.yaml')
if os.path.exists(data_yaml):
    import yaml
    with open(data_yaml,'r') as f:
        ydata = yaml.safe_load(f)
    names = ydata.get('names', {})
    # find images under ROBOFLOW_ROOT/train or images
    for root, dirs, files in os.walk(ROBOFLOW_ROOT):
        for fn in files:
            if fn.lower().endswith(('.jpg','.jpeg','.png')):
                src = os.path.join(root, fn)
                # find corresponding label .txt in labels/
                lbl = None
                lbl_path = os.path.join(root.replace('images','labels'), os.path.splitext(fn)[0]+'.txt')
                if os.path.exists(lbl_path):
                    lbl = lbl_path
                else:
                    alt = os.path.join(ROBOFLOW_ROOT, 'labels', os.path.splitext(fn)[0]+'.txt')
                    if os.path.exists(alt):
                        lbl = alt
                if lbl:
                    with open(lbl) as lf:
                        line = lf.readline().strip()
                        if line == '':
                            continue
                        cls_idx = int(line.split()[0])
                        clsname = names.get(cls_idx, str(cls_idx))
                else:
                    clsname = 'unknown'
                copy_unique(src, os.path.join(COMBINED_ROOT, clsname))

print("Combined ingestion finished. Classes:", sorted(os.listdir(COMBINED_ROOT)))

In [ ]:
import os, shutil
COMBINED = '/content/data/combined'
mapping = {
    'mobile_phone': 'mobile',
    'cellphone': 'mobile',
    'motherboard': 'circuit_board',
    'pcb': 'circuit_board',
    'battery_pack': 'battery',
    'charger_adapter': 'charger'
    # add as needed after inspecting os.listdir(COMBINED)
}

for src_name, dst_name in mapping.items():
    src = os.path.join(COMBINED, src_name)
    dst = os.path.join(COMBINED, dst_name)
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            shutil.move(os.path.join(src,f), os.path.join(dst,f))
        try:
            os.rmdir(src)
        except:
            pass

print("Post-normalization classes:", sorted(os.listdir(COMBINED)))

In [ ]:
import os, random, pandas as pd
from sklearn.model_selection import train_test_split
from shutil import copy2

random.seed(42)
COMBINED = '/content/data/combined'
OUT = '/content/data/splits'
TRAIN = os.path.join(OUT,'train')
VAL = os.path.join(OUT,'val')
TEST = os.path.join(OUT,'test')
for d in [TRAIN, VAL, TEST]:
    os.makedirs(d, exist_ok=True)

In [ ]:
rows = []
for cls in sorted(os.listdir(COMBINED)):
    cls_folder = os.path.join(COMBINED, cls)
    if not os.path.isdir(cls_folder):
        continue
    imgs = [os.path.join(cls_folder, f) for f in os.listdir(cls_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    n = len(imgs)
    if n == 0:
        continue
    if n < 10:
        # small class: 80/10/10 (best-effort)
        random.shuffle(imgs)
        cut1 = int(n*0.8)
        cut2 = cut1 + int(n*0.1)
        train_imgs = imgs[:cut1]
        val_imgs = imgs[cut1:cut2]
        test_imgs = imgs[cut2:]
    else:
        df = pd.DataFrame({'fp': imgs, 'cls': cls})
        train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['cls'], random_state=42)
        val_df, test_df = train_test_split(temp_df, test_size=0.3333, stratify=temp_df['cls'], random_state=42)
        train_imgs = train_df['fp'].tolist()
        val_imgs = val_df['fp'].tolist()
        test_imgs = test_df['fp'].tolist()
    # copy
    for p in train_imgs:
        dest = os.path.join(TRAIN, cls)
        os.makedirs(dest, exist_ok=True)
        copy2(p, dest)
        rows.append({'filepath': os.path.join('train', cls, os.path.basename(p)), 'class': cls, 'subset': 'train'})
    for p in val_imgs:
        dest = os.path.join(VAL, cls)
        os.makedirs(dest, exist_ok=True)
        copy2(p, dest)
        rows.append({'filepath': os.path.join('val', cls, os.path.basename(p)), 'class': cls, 'subset': 'val'})
    for p in test_imgs:
        dest = os.path.join(TEST, cls)
        os.makedirs(dest, exist_ok=True)
        copy2(p, dest)
        rows.append({'filepath': os.path.join('test', cls, os.path.basename(p)), 'class': cls, 'subset': 'test'})

manifest = pd.DataFrame(rows)
manifest.to_csv(os.path.join(OUT,'manifest_combined.csv'), index=False)
print("Splits created. Manifest saved to", os.path.join(OUT,'manifest_combined.csv'))

In [ ]:
import pandas as pd, os
manifest = pd.read_csv('/content/data/splits/manifest_combined.csv')
print("Total rows:", len(manifest))
print(manifest.groupby(['subset','class']).size().unstack(fill_value=0).head(50))
# quick sample count per subset
for s in ['train','val','test']:
    print(s, sum(manifest['subset']==s))
# sample images
from PIL import Image
import matplotlib.pyplot as plt
cls_list = manifest[manifest['subset']=='train']['class'].value_counts().index.tolist()[:4]
plt.figure(figsize=(12,6))
i=1
for cls in cls_list:
    p = os.path.join('/content/data/splits/train', cls, os.listdir(os.path.join('/content/data/splits/train',cls))[0])
    img = Image.open(p).convert('RGB')
    plt.subplot(1,4,i); plt.imshow(img); plt.title(cls); plt.axis('off'); i+=1
plt.show()

In [ ]:
import hashlib, glob, pandas as pd
rows=[]
for subset in ['train','val','test']:
    for cls in os.listdir(os.path.join('/content/data/splits', subset)):
        folder = os.path.join('/content/data/splits', subset, cls)
        if not os.path.isdir(folder): continue
        for f in os.listdir(folder):
            p = os.path.join(folder,f)
            with open(p,'rb') as fh:
                h = hashlib.sha256(fh.read()).hexdigest()
            rows.append({'subset':subset,'class':cls,'file':p,'sha256':h})
pd.DataFrame(rows).to_csv('/content/data/splits/images_with_hashes.csv', index=False)
print("Wrote images_with_hashes.csv")